In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("config_catalog_name", "", "config_catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("table_name", "", "table_name")
dbutils.widgets.text("email_recipient", "", "email_recipient")
dbutils.widgets.text("email_sender", "no-reply@databricks.com", "email_sender")
dbutils.widgets.dropdown("status", "1", ["0", "1"], "status")
dbutils.widgets.text("job_id", "", "job_id")
dbutils.widgets.text("job_name", "", "job_name")
dbutils.widgets.text("job_run_id", "", "job_run_id")

config_catalog_name = dbutils.widgets.get("config_catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
table_name = dbutils.widgets.get("table_name")
email_recipient = dbutils.widgets.get("email_recipient").split(',')
email_sender = dbutils.widgets.get("email_sender")
status = dbutils.widgets.get("status")
job_id = dbutils.widgets.get("job_id")
job_name = dbutils.widgets.get("job_name")
job_run_id = dbutils.widgets.get("job_run_id")

print("config_catalog_name: ", config_catalog_name)
print("config_schema_name: ", config_schema_name)
print("table_name == ", table_name)
print("email_recipient == ", email_recipient)
print("email_sender == ", email_sender)
print("status == ", status)
print("job_id == ", job_id)
print("job_name == ", job_name)
print("job_run_id == ", job_run_id)

In [0]:
import configparser
import json
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

import pandas as pd
from dbruntime.databricks_repl_context import get_context

In [0]:
config = configparser.ConfigParser()
config.read('setup.config')

In [0]:
def get_notebook_run_links(success=True):
    try:
        ctx = get_context()
        browser_host = getattr(ctx, "browserHostName", None)
        workspace_id = getattr(ctx, "workspaceId", "N/A")
        workspace_url = f"https://{browser_host}" if browser_host else None

        print("Workspace URL:", workspace_url)
        print("Workspace ID:", workspace_id)
        
        # 3. Build Safe HTML Link Anchors
        workspace_link = f"<a href='{workspace_url}/?o={workspace_id}' style='color: #1a73e8; text-decoration: none;'>{browser_host} [{workspace_id}]</a>"
        
        if job_id and job_run_id:
            job_link = f"<a href='{workspace_url}/#job/{job_id}/run/{job_run_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        elif job_id:
            job_link = f"<a href='{workspace_url}/#job/{job_id}' style='color: #1a73e8; text-decoration: none;'>{job_name} [{job_id}]</a>"
        else:
            job_link = "Interactive Notebook (Not a Job Run)"

        return {
            "Workspace": workspace_link,
            "Job": job_link,
            "Job Run": str(job_run_id) if job_run_id else "N/A",
            "Status": "Succeeded" if success else "Failed"
        }
    except Exception as e:
        print(f"Error fetching notebook run details: {e}")
        return {
            "Workspace": "Local Development (No Link)",
            "Job": "Manual Run (No Link)",
            "Job Run": "Local Testing",
            "Status": "Succeeded"
        }

In [0]:
def convert_df_to_csv_attachment(df: pd.DataFrame, filename: str) -> MIMEApplication:
    """
    Converts a pandas DataFrame into an in-memory CSV MIME attachment.
    """
    # Convert dataframe to CSV string without saving to disk
    csv_data = df.to_csv(index=False)
    
    # Create the attachment object
    attachment = MIMEApplication(csv_data, _subtype="csv")
    attachment.add_header(
        "Content-Disposition",
        "attachment",
        filename=filename
    )
    return attachment

In [0]:
def fetch_dqx_mappings():
    query = f"""
    WITH ranked_rules AS (
        SELECT 
            r.rule_id AS rule_id,
            r.rule_function AS rule_function,
            r.rule_dimension AS rule_dimension,
            r.rule_name AS rule_name,
            r.description AS rule_description,
            m.criticality AS criticality,
            m.arguments AS arguments,
            m.is_active AS is_active,
            m.column_name AS column,
            ROW_NUMBER() OVER (PARTITION BY m.table_name, m.column_name, r.rule_function ORDER BY m.updated_at DESC) as row_num
        FROM {config_catalog_name}.{config_schema_name}.dqx_rule_mappings m
        JOIN {config_catalog_name}.{config_schema_name}.dqx_rule_definitions r ON m.rule_id = r.rule_id
        WHERE m.table_name = '{table_name}' AND m.is_active = true
    ) 
    SELECT column, rule_name, rule_function, criticality, arguments FROM ranked_rules
    WHERE row_num = 1 
    and (column is not null and trim(column) != '')
    """
    df = spark.sql(query)
    return df.toPandas()

In [0]:
def send_completion_email(table_name, sender_email, recipient_email, config, run_details, success=True):
    # --- Configuration ---
    app_password = config.get('SMTP', 'password')
    smtp_server = config.get('SMTP', 'smtp_server')
    smtp_port = config.getint('SMTP', 'smtp_port')

    # --- Determine Status and Colors ---
    if success:
        status_text = "SUCCESS"
        status_color = "#34a853"  # Google green
        message = "A run of this job has completed successfully"
    else:
        status_text = "FAILURE"
        status_color = "#d93025"  # Google red
        message = "A run of this job has failed"

    subject = f"DQX Check {status_text.capitalize()} for Table | {table_name}"

    # --- Dynamic Rows Generation for the Box ---
    table_rows = ""
    for key, value in run_details.items():
        table_rows += f"""
        <tr>
            <td style='padding: 8px 0; font-weight: bold; color: #5f6368; width: 30%;'>{key}</td>
            <td style='padding: 8px 0; color: #202124;'>{value}</td>
        </tr>
        """

    # --- HTML Body Construction ---
    body = f"""
    <html>
    <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
        
        <!-- Header Section -->
        <h2 style="color: {status_color}; font-size: 24px; margin-bottom: 5px;">
            {message}
        </h2>
        <h3 style="font-size: 20px; color: #202124; margin-top: 0; margin-bottom: 20px;">
            Run details:
        </h3>
        
        <!-- Square Box Container -->
        <div style="border: 2px solid #dadce0; border-radius: 8px; padding: 20px; max-width: 600px; background-color: #f8f9fa;">
            <table style="width: 100%; border-collapse: collapse; font-size: 14px;">
                <tbody>
                    <tr>
                        <td style='padding: 8px 0; font-weight: bold; color: #5f6368; width: 30%;'>Table Target</td>
                        <td style='padding: 8px 0; color: #202124; font-weight: bold;'>{table_name}</td>
                    </tr>
                    {table_rows}
                </tbody>
            </table>
        </div>
        
    </body>
    </html>
    """

    msg = MIMEMultipart()
    msg['Subject'] = subject
    msg['From'] = sender_email
    msg['To'] = ','.join(recipient_email)
    msg.attach(MIMEText(body, 'html'))

    # --- CSV Attachment Logic using Helper Function ---
    try:
        df = fetch_dqx_mappings()
        csv_filename = f"{table_name}_dqx_report.csv"
        attachment = convert_df_to_csv_attachment(df, csv_filename)
        msg.attach(attachment)
    except Exception as csv_err:
        print(f"Error generating CSV attachment: {csv_err}")
        
    try:
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()
            server.login(sender_email, app_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        print("Email sent successfully!")
    except Exception as e:
        print(f"Error: {e}")
        raise


In [0]:
run_details_dict = get_notebook_run_links(success=(status == "1"))

In [0]:
send_completion_email(
    table_name, 
    email_sender , 
    email_recipient, 
    config ,
    run_details=run_details_dict,
    success=(status == "1")
)